# **Baseline Comparison for Microservices Anomaly Detection.**



Baselines implemented:
  1.  Statistical (Z-Score / 3-sigma rule)
  2.  CUSUM (Cumulative Sum Control Chart)
  3.  Isolation Forest                  [Liu et al., 2008]
  4.  One-Class SVM                     [Schölkopf et al., 1999]
  5.  Local Outlier Factor (LOF)        [Breunig et al., 2000]
  6.  AutoEncoder (MLP)                 [Sakurada & Yairi, 2014]
  7.  LSTM AutoEncoder                  [Malhotra et al., 2016]
  8.  Transformer AutoEncoder           [Vaswani et al., 2017 / xu et al., 2022]
  9.  Proposed: BiLSTM + Attention      (this paper)

Evaluation Protocol (IEEE / TNSM / TSC standard):
  - Identical train / val / test splits across all methods
  - Metrics: Precision, Recall, F1, ROC-AUC, PR-AUC
  - Wilcoxon signed-rank test for statistical significance
  - Mean ± Std over N_RUNS independent trials
  - LaTeX-ready result table printed to stdout
  - Publication-quality comparison figure saved to OUTPUT_DIR

**Import Libraries**

In [ ]:
import argparse
import warnings
import time
from copy import deepcopy
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import wilcoxon
from scipy.signal import find_peaks
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import networkx as nx

warnings.filterwarnings('ignore')

Re-use data generator and BiLSTM from the main paper file.
If running standalone, the classes are duplicated here for self-containment.

In [ ]:
try:
    from MicroservicesRCAPaper import (
        Config, MicroservicesDataGenerator, BiLSTMAnomalyDetector,
        prepare_sequences, train_model
    )
    _IMPORTED = True
except ImportError:
    _IMPORTED = False


Standalone fallbacks (identical to paper, condensed)

In [ ]:
if not _IMPORTED:
    class Config:
        N_SERVICES   = 50
        N_METRICS    = 6
        N_TIMESTAMPS = 5000
        RANDOM_SEED  = 42
        HIDDEN_DIM   = 128
        BATCH_SIZE   = 64
        LEARNING_RATE = 1e-3
        MAX_EPOCHS   = 30
        EARLY_STOP_PATIENCE = 7
        WINDOW_SIZE  = 60
        STRIDE       = 15
        FIGURE_DPI   = 300
        OUTPUT_DIR   = Path("/content/user-data/outputs")

        @classmethod
        def set_seed(cls):
            np.random.seed(cls.RANDOM_SEED)
            torch.manual_seed(cls.RANDOM_SEED)

    class MicroservicesDataGenerator:
        def __init__(self, n_services=50, n_metrics=6, n_timestamps=5000, seed=42):
            self.n_services  = n_services
            self.n_metrics   = n_metrics
            self.n_timestamps = n_timestamps
            np.random.seed(seed)
            self.tiers = {
                'frontend': list(range(0, 8)),
                'api':      list(range(8, 18)),
                'business': list(range(18, 35)),
                'data':     list(range(35, 43)),
                'database': list(range(43, 50))
            }
            self.causal_graph  = self._create_causal_graph()
            self.metric_names  = ['cpu','memory','latency','throughput','errors','connections']

        def _create_causal_graph(self):
            G = nx.DiGraph()
            G.add_nodes_from(range(self.n_services))
            for fe in self.tiers['frontend']:
                for t in np.random.choice(self.tiers['api'], size=np.random.randint(2,4), replace=False):
                    G.add_edge(fe, t)
            for api in self.tiers['api']:
                for t in np.random.choice(self.tiers['business'], size=np.random.randint(3,6), replace=False):
                    G.add_edge(api, t)
            for bus in self.tiers['business']:
                if np.random.rand() > 0.2:
                    for t in np.random.choice(self.tiers['data'], size=np.random.randint(1,3), replace=False):
                        G.add_edge(bus, t)
            for d in self.tiers['data']:
                for t in np.random.choice(self.tiers['database'], size=np.random.randint(1,2), replace=False):
                        G.add_edge(d, t)
            return G

        def generate(self):
            data = self._gen_baseline()
            labels, _ = self._inject_failures(data)
            self._inject_cps(data)
            data = (data - data.mean(axis=(0,1))) / (data.std(axis=(0,1)) + 1e-8)
            edge_index = np.array(list(self.causal_graph.edges())).T
            return {'data': data, 'edge_index': edge_index,
                    'anomaly_labels': labels, 'tiers': self.tiers,
                    'causal_graph': self.causal_graph}

        def _gen_baseline(self):
            t = np.arange(self.n_timestamps)
            base = 50 + 20*np.sin(2*np.pi*t/288) + 10*np.sin(2*np.pi*t/2016) + 5*np.sin(2*np.pi*t/12)
            noise = np.random.randn(self.n_services, self.n_timestamps, self.n_metrics)*3
            return base[np.newaxis,:,np.newaxis] + noise

        def _inject_failures(self, data):
            labels = np.zeros((self.n_timestamps, self.n_services))
            for _ in range(75):
                root  = np.random.choice(self.tiers['frontend']+self.tiers['api']+self.tiers['business'])
                start = np.random.randint(500, self.n_timestamps-500)
                affected = [root] + list(nx.descendants(self.causal_graph, root))[:15]
                for i, s in enumerate(affected):
                    delay = i * np.random.randint(2,6)
                    dur   = np.random.randint(40,100)
                    if start+delay+dur >= self.n_timestamps: continue
                    mag = (3.5+np.random.rand()*3.0)*(1-i*0.05)
                    data[s, start+delay:start+delay+dur, :] *= mag
                    labels[start+delay:start+delay+dur, s] = 1
            return labels, []

        def _inject_cps(self, data):
            for cp_id in range(8):
                ts = 800 + cp_id*500
                for s in np.random.choice(self.n_services, size=np.random.randint(15,30), replace=False):
                    ct = np.random.choice(['mean_shift','variance_change','both'])
                    if ct in ['mean_shift','both']:
                        data[s,ts:,:] *= np.random.uniform(2.0,3.0)
                    if ct in ['variance_change','both']:
                        mv = data[s,ts:,:].mean(axis=0,keepdims=True)
                        data[s,ts:,:] = mv + (data[s,ts:,:]-mv)*np.random.uniform(2.5,5.0)

    def prepare_sequences(data, labels, window_size=60, stride=15):
        n_s, n_t, n_m = data.shape
        starts = np.arange(0, n_t-window_size, stride)
        n = n_s * len(starts)
        X  = np.empty((n, window_size, n_m), dtype=np.float32)
        y  = np.empty(n, dtype=np.float32)
        si = np.empty(n, dtype=np.int64)
        st = np.empty(n, dtype=np.int64)
        idx = 0
        for s in range(n_s):
            for start in starts:
                X[idx]  = data[s, start:start+window_size, :]
                y[idx]  = labels[start:start+window_size, s].max()
                si[idx] = s; st[idx] = start; idx+=1
        return X, y, si, st

    class BiLSTMAnomalyDetector(nn.Module):
        def __init__(self, n_metrics=6, hidden_dim=128):
            super().__init__()
            self.encoder    = nn.LSTM(n_metrics, hidden_dim, num_layers=6, batch_first=True,
                                      bidirectional=True, dropout=0.3)
            self.attention  = nn.Linear(hidden_dim*2, 1)
            self.decoder    = nn.LSTM(hidden_dim*2, hidden_dim, batch_first=True)
            self.output     = nn.Linear(hidden_dim, n_metrics)
            self.classifier = nn.Sequential(
                nn.Linear(hidden_dim*2,128), nn.GELU(), nn.Dropout(0.3),
                nn.Linear(128,32), nn.GELU(), nn.Dropout(0.2),
                nn.Linear(32,1), nn.Sigmoid())
        def forward(self, x):
            enc, _ = self.encoder(x)
            aw  = F.softmax(self.attention(enc), dim=1)
            ctx = (enc * aw).sum(dim=1)
            dec_in  = ctx.unsqueeze(1).repeat(1, x.size(1), 1)
            dec, _  = self.decoder(dec_in)
            recon   = self.output(dec)
            score   = self.classifier(ctx).squeeze(-1)
            return {'reconstruction': recon, 'anomaly_score': score,
                    'attention': aw, 'encoding': ctx}
        def compute_loss(self, outputs, inputs, labels):
            rl = F.mse_loss(outputs['reconstruction'], inputs)
            pw = ((labels==0).sum()/(labels==1).sum().clamp(min=1.0)).clamp(max=10.0)
            bl = F.binary_cross_entropy(outputs['anomaly_score'], labels,
                     weight=torch.where(labels==1, pw, torch.ones_like(labels)))
            return {'total': rl + 0.3*bl, 'reconstruction': rl, 'classification': bl}

    def train_model(model, X_tr, y_tr, X_v, y_v,
                    epochs=30, batch_size=64, lr=1e-3):
        from torch.utils.data import TensorDataset, DataLoader
        device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model   = model.to(device)
        opt     = torch.optim.Adam(model.parameters(), lr=lr)
        sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)
        tr_ds   = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr))
        v_ds    = TensorDataset(torch.FloatTensor(X_v),  torch.FloatTensor(y_v))
        pin     = device.type == 'cuda'
        tr_dl   = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                             num_workers=2, pin_memory=pin)
        v_dl    = DataLoader(v_ds,  batch_size=batch_size*2, shuffle=False,
                             num_workers=2, pin_memory=pin)
        best_vl = float('inf'); pat = 0; best_sd = None
        for ep in range(epochs):
            model.train()
            tl = 0; nb = 0
            for bx, by in tr_dl:
                bx, by = bx.to(device), by.to(device)
                out = model(bx); loss = model.compute_loss(out, bx, by)
                opt.zero_grad(); loss['total'].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); tl += loss['total'].item(); nb += 1
            model.eval(); vl = 0; nvb = 0
            with torch.no_grad():
                for bx, by in v_dl:
                    bx, by = bx.to(device), by.to(device)
                    out = model(bx); loss = model.compute_loss(out, bx, by)
                    vl += loss['total'].item(); nvb += 1
            tl /= nb; vl /= nvb; sched.step(vl)
            if vl < best_vl:
                best_vl = vl; pat = 0
                best_sd = {k: v.cpu() for k, v in model.state_dict().items()}
            else:
                pat += 1
                if pat >= 7: break
        if best_sd: model.load_state_dict(best_sd)
        return model.cpu()

Hyperparameter overrides for comparison study

In [ ]:
OUTPUT_DIR = Path("/content/user-data/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_RUNS = 1          # independent trials for mean ± std
ALPHA  = 0.05       # Wilcoxon significance level

**Baseline Implementations**

1.  Statistical Baseline (Z-Score / 3-sigma)

In [ ]:
class StatisticalDetector:
    """
    Univariate Z-Score anomaly detector.
    A window is anomalous if any feature's z-score exceeds a threshold.
    Reference: Shewhart (1931), widely used in SRE / AIOps literature.
    """
    def __init__(self, threshold: float = 3.0):
        self.threshold = threshold
        self.mean_  = None
        self.std_   = None

    def fit(self, X: np.ndarray) -> 'StatisticalDetector':
        # X shape: (n_samples, window, n_metrics)
        flat = X.reshape(-1, X.shape[-1])
        self.mean_ = flat.mean(axis=0)
        self.std_  = flat.std(axis=0) + 1e-8
        return self

    def score_samples(self, X: np.ndarray) -> np.ndarray:
        flat = X.reshape(len(X), -1, X.shape[-1])
        z    = np.abs((flat - self.mean_) / self.std_)
        return z.max(axis=(1, 2))   # worst-case z-score per window

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        raw = self.score_samples(X)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)

2. CUSUM

In [ ]:
class CUSUMDetector:
    """
    Cumulative-Sum (CUSUM) control chart.
    Reference: Page (1954); popularised for SRE by Brutlag (2000).
    """
    def __init__(self, k: float = 0.5, h: float = 5.0):
        self.k = k   # allowance
        self.h = h   # decision interval
        self.mean_ = None
        self.std_  = None

    def fit(self, X: np.ndarray) -> 'CUSUMDetector':
        flat = X.reshape(-1, X.shape[-1])
        self.mean_ = flat.mean(axis=0)
        self.std_  = flat.std(axis=0) + 1e-8
        return self

    def score_samples(self, X: np.ndarray) -> np.ndarray:
        # For each window compute max CUSUM statistic over features
        # X: (n, T, F)
        scores = np.zeros(len(X))
        for i, seq in enumerate(X):
            z = (seq - self.mean_) / self.std_
            S_pos = np.zeros(z.shape[-1])
            S_neg = np.zeros(z.shape[-1])
            max_s = 0.0
            for t in range(len(seq)):
                S_pos = np.maximum(0, S_pos + z[t] - self.k)
                S_neg = np.maximum(0, S_neg - z[t] - self.k)
                max_s = max(max_s, S_pos.max(), S_neg.max())
            scores[i] = max_s
        return scores

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        raw = self.score_samples(X)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)

3. Isolation Forest

In [ ]:
class IsolationForestDetector:
    """
    Isolation Forest for multivariate anomaly detection.
    Reference: Liu, Ting & Zhou (2008). Isolation Forest. ICDM.
    """
    def __init__(self, n_estimators: int = 200, contamination: float = 0.1,
                 random_state: int = 42):
        self.model = IsolationForest(
            n_estimators=n_estimators,
            contamination=contamination,
            random_state=random_state,
            n_jobs=-1
        )
        self.scaler = StandardScaler()

    def _flatten(self, X: np.ndarray) -> np.ndarray:
        return X.reshape(len(X), -1)

    def fit(self, X: np.ndarray) -> 'IsolationForestDetector':
        flat = self._flatten(X)
        flat = self.scaler.fit_transform(flat)
        self.model.fit(flat)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        flat  = self.scaler.transform(self._flatten(X))
        raw   = -self.model.score_samples(flat)   # higher → more anomalous
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)

4.  One-Class SVM

In [ ]:
class OneClassSVMDetector:
    """
    One-Class SVM (ν-SVM) for novelty detection.
    Reference: Schölkopf et. al (1999). Estimating the support of a
               high-dimensional distribution. Neural Computation.
    Note: PCA projection used to manage dimensionality.
    """
    def __init__(self, nu: float = 0.1, kernel: str = 'rbf',
                 gamma: str = 'scale', n_components: int = 20):
        from sklearn.decomposition import PCA
        self.model  = OneClassSVM(nu=nu, kernel=kernel, gamma=gamma)
        self.scaler = StandardScaler()
        self.pca    = PCA(n_components=n_components, random_state=42)
        self.n_components = n_components

    def _flatten(self, X: np.ndarray) -> np.ndarray:
        return X.reshape(len(X), -1)

    def fit(self, X: np.ndarray) -> 'OneClassSVMDetector':
        flat = self.scaler.fit_transform(self._flatten(X))
        flat = self.pca.fit_transform(flat)
        self.model.fit(flat)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        flat = self.scaler.transform(self._flatten(X))
        flat = self.pca.transform(flat)
        raw  = -self.model.score_samples(flat)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)

5.  Local Outlier Factor

In [ ]:
class LOFDetector:
    """
    Local Outlier Factor (novelty=True for test-time scoring).
    Reference: Breunig, Kriegel, Ng & Sander (2000). LOF: Identifying
               density-based local outliers. SIGMOD.
    """
    def __init__(self, n_neighbors: int = 20, n_components: int = 20):
        from sklearn.decomposition import PCA
        self.model  = LocalOutlierFactor(n_neighbors=n_neighbors,
                                         novelty=True, n_jobs=-1)
        self.scaler = StandardScaler()
        self.pca    = PCA(n_components=n_components, random_state=42)

    def _flatten(self, X: np.ndarray) -> np.ndarray:
        return X.reshape(len(X), -1)

    def fit(self, X: np.ndarray) -> 'LOFDetector':
        flat = self.scaler.fit_transform(self._flatten(X))
        flat = self.pca.fit_transform(flat)
        self.model.fit(flat)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        flat = self.scaler.transform(self._flatten(X))
        flat = self.pca.transform(flat)
        raw  = -self.model.score_samples(flat)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)

6.  MLP AutoEncoder

In [ ]:
class MLPAutoEncoder(nn.Module):
    """
    Fully-connected (MLP) AutoEncoder for anomaly detection via
    reconstruction error.
    Reference: Sakurada & Yairi (2014). Anomaly detection using
               autoencoders with nonlinear dimensionality reduction. MLSDA.
    """
    def __init__(self, input_dim: int, hidden_dims: List[int] = None,
                 latent_dim: int = 64):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [256, 128, 64]

        enc_layers: List[nn.Module] = []
        prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.2)]
            prev = h
        enc_layers += [nn.Linear(prev, latent_dim)]
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers: List[nn.Module] = []
        prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.2)]
            prev = h
        dec_layers += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))

class MLPAutoEncoderDetector:
    def __init__(self, window_size: int = 60, n_metrics: int = 6,
                 latent_dim: int = 64, epochs: int = 30, batch_size: int = 64,
                 lr: float = 1e-3):
        self.input_dim  = window_size * n_metrics
        self.latent_dim = latent_dim
        self.epochs     = epochs
        self.batch_size = batch_size
        self.lr         = lr
        self.model      = None
        self.device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.scaler     = StandardScaler()

    def fit(self, X: np.ndarray, X_val: Optional[np.ndarray] = None) -> 'MLPAutoEncoderDetector':
        from torch.utils.data import TensorDataset, DataLoader
        flat = self.scaler.fit_transform(X.reshape(len(X), -1))
        self.model = MLPAutoEncoder(self.input_dim, latent_dim=self.latent_dim).to(self.device)
        opt   = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)
        ds    = TensorDataset(torch.FloatTensor(flat))
        dl    = DataLoader(ds, batch_size=self.batch_size, shuffle=True,
                           num_workers=2, pin_memory=self.device.type=='cuda')
        best_loss = float('inf'); best_sd = None; pat = 0
        for ep in range(self.epochs):
            self.model.train(); ep_loss = 0; nb = 0
            for (bx,) in dl:
              bx = bx.to(self.device)

              # Add Gaussian noise
              noise = 0.01 * torch.randn_like(bx)
              noisy_bx = bx + noise

              out = self.model(noisy_bx)
              loss = F.mse_loss(out, bx)  # target remains CLEAN bx

              opt.zero_grad()
              loss.backward()
              opt.step()
              ep_loss += loss.item(); nb += 1
            ep_loss /= nb; sched.step()
            if ep_loss < best_loss:
                best_loss = ep_loss; pat = 0
                best_sd = {k: v.cpu() for k, v in self.model.state_dict().items()}
            else:
                pat += 1
                if pat >= 10: break
        if best_sd: self.model.load_state_dict(best_sd)
        self.model = self.model.cpu()
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        flat   = self.scaler.transform(X.reshape(len(X), -1))
        t_flat = torch.FloatTensor(flat)
        self.model.eval()
        errors = []
        with torch.no_grad():
            for i in range(0, len(t_flat), 512):
                bx  = t_flat[i:i+512]
                out = self.model(bx)
                err = ((out - bx) ** 2).mean(dim=1)
                errors.append(err.numpy())
        raw = np.concatenate(errors)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)

7.  LSTM AutoEncoder

In [ ]:
class LSTMAutoEncoderModel(nn.Module):
    """
    Sequence-to-Sequence LSTM AutoEncoder.
    Reference: Malhotra et al. (2016). LSTM-based Encoder-Decoder for
               Multi-sensor Anomaly Detection. ICML Anomaly Detection Workshop.
    """
    def __init__(self, n_metrics: int = 6, hidden_dim: int = 128, n_layers: int = 3):
        super().__init__()
        self.enc = nn.LSTM(n_metrics, hidden_dim, n_layers,
                           batch_first=True, dropout=0.1)
        self.dec = nn.LSTM(hidden_dim, hidden_dim, n_layers,
                           batch_first=True, dropout=0.1)
        self.out = nn.Linear(hidden_dim, n_metrics)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (h, c) = self.enc(x)
        seq_len   = x.size(1)
        dec_in    = h[-1].unsqueeze(1).repeat(1, seq_len, 1)
        dec_out, _ = self.dec(dec_in, (h, c))
        return self.out(dec_out)


class LSTMAutoEncoderDetector:
    def __init__(self, n_metrics: int = 6, hidden_dim: int = 128,
                 epochs: int = 30, batch_size: int = 64, lr: float = 1e-3):
        self.n_metrics  = n_metrics
        self.hidden_dim = hidden_dim
        self.epochs     = epochs
        self.batch_size = batch_size
        self.lr         = lr
        self.model      = None
        self.device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def fit(self, X: np.ndarray, X_val: Optional[np.ndarray] = None) -> 'LSTMAutoEncoderDetector':
        from torch.utils.data import TensorDataset, DataLoader
        self.model = LSTMAutoEncoderModel(self.n_metrics, self.hidden_dim).to(self.device)
        opt   = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)
        ds    = TensorDataset(torch.FloatTensor(X))
        dl    = DataLoader(ds, batch_size=self.batch_size, shuffle=True,
                           num_workers=2, pin_memory=self.device.type=='cuda')
        best_loss = float('inf'); best_sd = None; pat = 0
        for ep in range(self.epochs):
            self.model.train(); ep_loss = 0; nb = 0
            for (bx,) in dl:
                bx  = bx.to(self.device)
                out = self.model(bx)
                loss = F.mse_loss(out, bx)
                opt.zero_grad(); loss.backward(); opt.step()
                ep_loss += loss.item(); nb += 1
            ep_loss /= nb; sched.step()
            if ep_loss < best_loss:
                best_loss = ep_loss; pat = 0
                best_sd = {k: v.cpu() for k, v in self.model.state_dict().items()}
            else:
                pat += 1
                if pat >= 10: break
        if best_sd: self.model.load_state_dict(best_sd)
        self.model = self.model.cpu()
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.model.eval(); errors = []
        with torch.no_grad():
            for i in range(0, len(X), 256):
                bx  = torch.FloatTensor(X[i:i+256])
                out = self.model(bx)
                err = ((out - bx) ** 2).mean(dim=(1, 2))
                errors.append(err.numpy())
        raw = np.concatenate(errors)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)

8.  Transformer AutoEncoder

In [ ]:
class TransformerAutoEncoderModel(nn.Module):
    """
    Transformer-based AutoEncoder for time-series anomaly detection.
    Reference: Xu et al. (2022). Anomaly Transformer: Time Series Anomaly
               Detection with Association Discrepancy. ICLR.
               (simplified variant without association discrepancy for
               fair comparison as a reconstruction-only baseline)
    """
    def __init__(self, n_metrics: int = 6, d_model: int = 128,
                 nhead: int = 4, num_layers: int = 4,
                 dim_feedforward: int = 128, dropout: float = 0.1,
                 max_seq_len: int = 60):
        super().__init__()
        self.input_proj  = nn.Linear(n_metrics, d_model)
        self.pos_enc     = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.dec = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, n_metrics)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        emb     = self.input_proj(x) + self.pos_enc[:, :seq_len, :]
        mem     = self.transformer(emb)
        out     = self.dec(emb, mem)
        return self.output_proj(out)


class TransformerAutoEncoderDetector:
    def __init__(self, n_metrics: int = 6, d_model: int = 128, nhead: int = 4,
                 num_layers: int = 4, window_size: int = 60,
                 epochs: int = 30, batch_size: int = 64, lr: float = 1e-3):
        self.n_metrics   = n_metrics
        self.d_model     = d_model
        self.nhead       = nhead
        self.num_layers  = num_layers
        self.window_size = window_size
        self.epochs      = epochs
        self.batch_size  = batch_size
        self.lr          = lr
        self.model       = None
        self.device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def fit(self, X: np.ndarray, X_val: Optional[np.ndarray] = None) -> 'TransformerAutoEncoderDetector':
        from torch.utils.data import TensorDataset, DataLoader
        self.model = TransformerAutoEncoderModel(
            self.n_metrics, self.d_model, self.nhead,
            self.num_layers, max_seq_len=self.window_size
        ).to(self.device)
        opt   = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)
        ds    = TensorDataset(torch.FloatTensor(X))
        dl    = DataLoader(ds, batch_size=self.batch_size, shuffle=True,
                           num_workers=2, pin_memory=self.device.type=='cuda')
        best_loss = float('inf'); best_sd = None; pat = 0
        for ep in range(self.epochs):
            self.model.train(); ep_loss = 0; nb = 0
            for (bx,) in dl:
                bx  = bx.to(self.device)
                out = self.model(bx)
                loss = F.mse_loss(out, bx)
                opt.zero_grad(); loss.backward(); opt.step()
                ep_loss += loss.item(); nb += 1
            ep_loss /= nb; sched.step()
            if ep_loss < best_loss:
                best_loss = ep_loss; pat = 0
                best_sd = {k: v.cpu() for k, v in self.model.state_dict().items()}
            else:
                pat += 1
                if pat >= 10: break
        if best_sd: self.model.load_state_dict(best_sd)
        self.model = self.model.cpu()
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.model.eval(); errors = []
        with torch.no_grad():
            for i in range(0, len(X), 256):
                bx  = torch.FloatTensor(X[i:i+256])
                out = self.model(bx)
                err = ((out - bx) ** 2).mean(dim=(1, 2))
                errors.append(err.numpy())
        raw = np.concatenate(errors)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)

9.  Proposed: BiLSTM + Attention  (wrapper for unified interface)

In [ ]:
class ProposedDetector:
    """Thin wrapper so BiLSTMAnomalyDetector fits the Detector interface."""
    def __init__(self, n_metrics: int = 6, hidden_dim: int = 128,
                 epochs: int = 30, batch_size: int = 64, lr: float = 1e-3):
        self.n_metrics  = n_metrics
        self.hidden_dim = hidden_dim
        self.epochs     = epochs
        self.batch_size = batch_size
        self.lr         = lr
        self.model      = None
        self.device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def fit(self, X_tr: np.ndarray, X_val: Optional[np.ndarray] = None,
            y_tr: Optional[np.ndarray] = None,
            y_val: Optional[np.ndarray] = None) -> 'ProposedDetector':
        if y_tr is None:
            y_tr  = np.zeros(len(X_tr), dtype=np.float32)
        if y_val is None:
            y_val = np.zeros(len(X_val) if X_val is not None else 1, dtype=np.float32)
        if X_val is None:
            X_val = X_tr[:max(1, len(X_tr)//10)]
            y_val = y_tr[:max(1, len(y_tr)//10)]
        m = BiLSTMAnomalyDetector(self.n_metrics, self.hidden_dim)
        self.model = train_model(m, X_tr, y_tr, X_val, y_val,
                                 epochs=self.epochs,
                                 batch_size=self.batch_size,
                                 lr=self.lr)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.model.eval(); scores = []; recons = []
        with torch.no_grad():
            for i in range(0, len(X), 256):
                bx  = torch.FloatTensor(X[i:i+256])
                out = self.model(bx)
                scores.append(out['anomaly_score'].numpy())
                re  = ((out['reconstruction'] - bx)**2).mean(dim=(1,2))
                recons.append(re.numpy())
        s = np.concatenate(scores);  r = np.concatenate(recons)
        ns = (s - s.min())/(s.max() - s.min() + 1e-8)
        nr = (r - r.min())/(r.max() - r.min() + 1e-8)
        return 0.7*ns + 0.3*nr

**Evaluation helpers**

In [ ]:
def best_threshold_f1(y_true: np.ndarray, y_score: np.ndarray) -> float:
    best_f1, best_t = 0.0, 0.5
    for t in np.linspace(0.05, 0.95, 100):
        f1 = f1_score(y_true, (y_score > t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1; best_t = t
    return best_t

In [ ]:
def compute_metrics(y_true: np.ndarray, y_score: np.ndarray) -> Dict[str, float]:
    t    = best_threshold_f1(y_true, y_score)
    pred = (y_score > t).astype(int)
    roc  = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else 0.0
    prc  = average_precision_score(y_true, y_score) if len(np.unique(y_true)) > 1 else 0.0
    return {
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall':    recall_score(y_true, pred, zero_division=0),
        'f1':        f1_score(y_true, pred, zero_division=0),
        'roc_auc':   roc,
        'pr_auc':    prc
    }

**Main comparison loop**

In [ ]:
BASELINE_REGISTRY = {
    'Z-Score (3σ)':       lambda cfg: StatisticalDetector(threshold=3.0),
    'CUSUM':              lambda cfg: CUSUMDetector(),
    'Isolation Forest':   lambda cfg: IsolationForestDetector(n_estimators=200),
    'One-Class SVM':      lambda cfg: OneClassSVMDetector(),
    'LOF':                lambda cfg: LOFDetector(),
    'MLP AutoEncoder':    lambda cfg: MLPAutoEncoderDetector(
                              window_size=cfg.WINDOW_SIZE,
                              n_metrics=cfg.N_METRICS,
                              epochs=cfg.MAX_EPOCHS),
    'LSTM AutoEncoder':   lambda cfg: LSTMAutoEncoderDetector(
                              n_metrics=cfg.N_METRICS,
                              epochs=cfg.MAX_EPOCHS),
    'Transformer AE':     lambda cfg: TransformerAutoEncoderDetector(
                              n_metrics=cfg.N_METRICS,
                              window_size=cfg.WINDOW_SIZE,
                              epochs=cfg.MAX_EPOCHS),
    'Proposed (BiLSTM+A)': lambda cfg: ProposedDetector(
                              n_metrics=cfg.N_METRICS,
                              hidden_dim=cfg.HIDDEN_DIM,
                              epochs=cfg.MAX_EPOCHS),
}

METRIC_KEYS = ['precision', 'recall', 'f1', 'roc_auc', 'pr_auc']

In [ ]:
def run_comparison(n_runs: int = N_RUNS, quick: bool = False) -> Dict:
    """
    Run full baseline comparison over n_runs independent trials.

    Returns
    -------
    results : dict
        results[method][metric] = list of values over runs
    """
    cfg = Config()
    if quick:
        cfg.MAX_EPOCHS = 5
        n_runs = 1

    all_results: Dict[str, Dict[str, List[float]]] = {
        m: {k: [] for k in METRIC_KEYS} for m in BASELINE_REGISTRY
    }
    timing: Dict[str, List[float]] = {m: [] for m in BASELINE_REGISTRY}

    for run in range(n_runs):
        seed = cfg.RANDOM_SEED + run
        print(f"\n{'='*80}")
        print(f"  RUN {run+1}/{n_runs}  (seed={seed})")
        print(f"{'='*80}")

        # ── Data generation ──────────────────────────────────────────────────
        np.random.seed(seed); torch.manual_seed(seed)
        gen     = MicroservicesDataGenerator(
                      n_services=cfg.N_SERVICES,
                      n_metrics=cfg.N_METRICS,
                      n_timestamps=cfg.N_TIMESTAMPS,
                      seed=seed)
        dataset = gen.generate()

        X, y, sids, stimes = prepare_sequences(
            dataset['data'], dataset['anomaly_labels'],
            window_size=cfg.WINDOW_SIZE, stride=cfg.STRIDE)

        n = len(X)
        tr  = int(0.6*n); vl = int(0.2*n)
        X_tr,  y_tr  = X[:tr],          y[:tr]
        X_val, y_val = X[tr:tr+vl],     y[tr:tr+vl]
        X_te,  y_te  = X[tr+vl:],       y[tr+vl:]

        # ── Evaluate each baseline ────────────────────────────────────────────
        for name, factory in BASELINE_REGISTRY.items():
            print(f"\n  ▶ {name}")
            det = factory(cfg)
            t0  = time.time()

            try:
                # ── fit ──
                if name == 'Proposed (BiLSTM+A)':
                    det.fit(X_tr, X_val, y_tr, y_val)
                elif isinstance(det, (MLPAutoEncoderDetector,
                                      LSTMAutoEncoderDetector,
                                      TransformerAutoEncoderDetector)):
                    # Unsupervised: train only on normal windows
                    normal_mask = y_tr == 0
                    det.fit(X_tr[normal_mask])
                else:
                    # sklearn-style: fit on normal windows
                    normal_mask = y_tr == 0
                    det.fit(X_tr[normal_mask])

                # ── score test set ──
                scores = det.predict_proba(X_te)
                m_dict = compute_metrics(y_te, scores)

            except Exception as e:
                print(f"    ERROR: {e}")
                m_dict = {k: 0.0 for k in METRIC_KEYS}

            elapsed = time.time() - t0
            timing[name].append(elapsed)

            for k in METRIC_KEYS:
                all_results[name][k].append(m_dict[k])

            print(f"    F1={m_dict['f1']:.4f}  ROC-AUC={m_dict['roc_auc']:.4f}  "
                  f"PR-AUC={m_dict['pr_auc']:.4f}  ({elapsed:.1f}s)")

    return all_results, timing

Statistical significance testing

In [ ]:
def significance_test(
    all_results: Dict,
    proposed_key: str = 'Proposed (BiLSTM+A)',
    metric: str = 'f1',
    alpha: float = ALPHA
) -> Dict[str, Dict]:
    """
    Wilcoxon signed-rank test: proposed vs. each baseline on F1 scores.
    Returns dict of {baseline: {stat, p_value, significant, direction}}.
    """
    proposed_vals = np.array(all_results[proposed_key][metric])
    out = {}
    for name, res in all_results.items():
        if name == proposed_key:
            continue
        base_vals = np.array(res[metric])
        if len(proposed_vals) < 2 or np.allclose(proposed_vals, base_vals):
            out[name] = {'stat': 0.0, 'p_value': 1.0,
                         'significant': False, 'direction': '='}
            continue
        try:
            stat, p = wilcoxon(proposed_vals, base_vals, alternative='greater')
        except Exception:
            stat, p = 0.0, 1.0
        out[name] = {
            'stat':        float(stat),
            'p_value':     float(p),
            'significant': bool(p < alpha),
            'direction':   '↑' if p < alpha else ('↓' if proposed_vals.mean() < base_vals.mean() else '=')
        }
    return out

Comparison figure

In [ ]:
def plot_comparison(all_results: Dict, sig_tests: Dict, timing: Dict):
    """
    Generate a 3-panel IEEE-style figure:
      (a) Grouped bar chart: F1, ROC-AUC, PR-AUC
      (b) Radar / spider chart across all five metrics
      (c) F1 improvement over best baseline (proposed vs. others)
    """
    plt.rcParams.update({
        'font.family':     'serif',
        'font.size':       10,
        'axes.titlesize':  11,
        'axes.labelsize':  10,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 9,
        'figure.dpi':      Config.FIGURE_DPI
    })
    sns.set_palette("tab10")

    methods  = list(all_results.keys())
    proposed = 'Proposed (BiLSTM+A)'
    colors   = sns.color_palette("tab10", len(methods))
    m_colors = {m: colors[i] for i, m in enumerate(methods)}

    f1_means    = [np.mean(all_results[m]['f1'])      for m in methods]
    roc_means   = [np.mean(all_results[m]['roc_auc']) for m in methods]
    pr_means    = [np.mean(all_results[m]['pr_auc'])  for m in methods]
    f1_stds     = [np.std(all_results[m]['f1'])       for m in methods]
    roc_stds    = [np.std(all_results[m]['roc_auc'])  for m in methods]
    pr_stds     = [np.std(all_results[m]['pr_auc'])   for m in methods]

    fig = plt.figure(figsize=(18, 12))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

    # ── (a) Grouped bar: F1 ──────────────────────────────────────────────────
    ax_f1 = fig.add_subplot(gs[0, :2])
    x    = np.arange(len(methods))
    bars = ax_f1.bar(x, f1_means, yerr=f1_stds,
                     color=[m_colors[m] for m in methods],
                     capsize=4, edgecolor='black', linewidth=0.6,
                     error_kw={'elinewidth': 1.2})

    # Annotate significance
    prop_idx = methods.index(proposed)
    for i, m in enumerate(methods):
        if m == proposed:
            bars[i].set_edgecolor('black')
            bars[i].set_linewidth(2.0)
        val = f1_means[i]
        ax_f1.text(i, val + f1_stds[i] + 0.005,
                   f'{val:.3f}', ha='center', va='bottom', fontsize=8)
        if m != proposed and m in sig_tests and sig_tests[m]['significant']:
            ax_f1.text(i, val + f1_stds[i] + 0.022, '*',
                       ha='center', va='bottom', fontsize=13, color='red')

    ax_f1.set_xticks(x)
    ax_f1.set_xticklabels(methods, rotation=30, ha='right')
    ax_f1.set_ylim([0, min(1.05, max(f1_means) + 0.12)])
    ax_f1.set_ylabel('F1 Score')
    ax_f1.set_title('(a) F1 Score Comparison', fontweight='bold')
    ax_f1.axhline(f1_means[prop_idx], color='black', linestyle='--',
                  linewidth=1, alpha=0.5, label='Proposed F1')
    ax_f1.legend(loc='lower right')
    ax_f1.grid(axis='y', alpha=0.3)

    # ── (b) ROC-AUC bar ──────────────────────────────────────────────────────
    ax_roc = fig.add_subplot(gs[0, 2])
    ax_roc.barh(methods, roc_means, xerr=roc_stds,
                color=[m_colors[m] for m in methods],
                capsize=3, edgecolor='black', linewidth=0.6)
    ax_roc.set_xlim([0, 1.05])
    ax_roc.set_xlabel('ROC-AUC')
    ax_roc.set_title('(b) ROC-AUC', fontweight='bold')
    ax_roc.axvline(roc_means[prop_idx], color='black',
                   linestyle='--', linewidth=1, alpha=0.5)
    ax_roc.grid(axis='x', alpha=0.3)

    # ── (c) Radar chart ──────────────────────────────────────────────────────
    ax_radar = fig.add_subplot(gs[1, 0], polar=True)
    metric_labels = ['Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']
    N   = len(metric_labels)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    # Plot a subset of representative methods to avoid clutter
    radar_methods = ['Z-Score (3σ)', 'Isolation Forest',
                     'LSTM AutoEncoder', 'Transformer AE', proposed]
    for m in radar_methods:
        if m not in all_results: continue
        vals = [np.mean(all_results[m][k]) for k in METRIC_KEYS]
        vals += vals[:1]
        ax_radar.plot(angles, vals, 'o-', linewidth=1.8,
                      color=m_colors[m], label=m)
        ax_radar.fill(angles, vals, alpha=0.05, color=m_colors[m])

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(metric_labels, size=9)
    ax_radar.set_ylim(0, 1)
    ax_radar.set_title('(c) Radar: Selected Methods', fontweight='bold', pad=15)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(1.45, 1.15), fontsize=8)

    # ── (d) F1 gain over each baseline ───────────────────────────────────────
    ax_gain = fig.add_subplot(gs[1, 1])
    baselines   = [m for m in methods if m != proposed]
    gains       = [np.mean(all_results[proposed]['f1']) -
                   np.mean(all_results[b]['f1']) for b in baselines]
    gain_colors = ['#2ecc71' if g > 0 else '#e74c3c' for g in gains]
    bars2 = ax_gain.barh(baselines, gains, color=gain_colors,
                         edgecolor='black', linewidth=0.6)
    ax_gain.axvline(0, color='black', linewidth=0.8)
    ax_gain.set_xlabel('F1 Gain (Proposed − Baseline)')
    ax_gain.set_title('(d) F1 Improvement over Baselines', fontweight='bold')
    ax_gain.grid(axis='x', alpha=0.3)
    for bar, g in zip(bars2, gains):
        ax_gain.text(g + (0.001 if g >= 0 else -0.001),
                     bar.get_y() + bar.get_height()/2,
                     f'{g:+.3f}', va='center',
                     ha='left' if g >= 0 else 'right', fontsize=8)

    # ── (e) Inference time ────────────────────────────────────────────────────
    ax_time = fig.add_subplot(gs[1, 2])
    t_means = [np.mean(timing[m]) for m in methods]
    ax_time.bar(range(len(methods)), t_means,
                color=[m_colors[m] for m in methods],
                edgecolor='black', linewidth=0.6)
    ax_time.set_xticks(range(len(methods)))
    ax_time.set_xticklabels(methods, rotation=40, ha='right')
    ax_time.set_ylabel('Wall-Clock Time (s)')
    ax_time.set_title('(e) Training + Inference Time', fontweight='bold')
    ax_time.set_yscale('log')
    ax_time.grid(axis='y', alpha=0.3)

    plt.suptitle(
        'IEEE Baseline Comparison — Microservices Anomaly Detection\n'
        r'($\dagger$ $p<0.05$, Wilcoxon signed-rank test)',
        fontsize=13, fontweight='bold', y=1.01
    )

    out_path = OUTPUT_DIR / 'baseline_comparison.png'
    plt.savefig(out_path, dpi=Config.FIGURE_DPI, bbox_inches='tight')
    plt.close()
    print(f"\n✓ Comparison figure saved → {out_path}")

Per-metric summary table

In [ ]:
def print_full_results_table(all_results: Dict):
    header = f"{'Method':<24}"
    for k in METRIC_KEYS:
        header += f"  {k.upper():>14}"
    print("\n" + "="*96)
    print("FULL METRIC BREAKDOWN  (Mean ± Std)")
    print("="*96)
    print(header)
    print("-"*96)
    for name, res in all_results.items():
        row = f"{name:<24}"
        for k in METRIC_KEYS:
            mu = np.mean(res[k]); sd = np.std(res[k])
            row += f"  {mu:.4f}±{sd:.4f}"
        print(row)

Entry point

In [ ]:
def main():
    parser = argparse.ArgumentParser(
        description='IEEE Baseline Comparison for Microservices Anomaly Detection')
    parser.add_argument('--quick', action='store_true',
                        help='Quick smoke-test: 1 run, 5 epochs')
    parser.add_argument('--runs', type=int, default=N_RUNS,
                        help=f'Number of independent runs (default: {N_RUNS})')
    args = parser.parse_args([])

    Config.set_seed()
    print("=" * 80)
    print("IEEE BASELINE COMPARISON — MICROSERVICES ANOMALY DETECTION")
    print("=" * 80)
    print(f"  Runs:     {1 if args.quick else args.runs}")
    print(f"  Baselines:{len(BASELINE_REGISTRY)}")
    print(f"  Metrics:  {', '.join(METRIC_KEYS)}")
    print(f"  Test:     Wilcoxon signed-rank (α={ALPHA})")
    print()

    all_results, timing = run_comparison(
        n_runs=args.runs, quick=args.quick)

    proposed = 'Proposed (BiLSTM+A)'
    sig_tests = significance_test(all_results, proposed_key=proposed, metric='f1')

    print_full_results_table(all_results)
    print_latex_table(all_results, sig_tests, timing)
    plot_comparison(all_results, sig_tests, timing)

    print("\n" + "="*80)
    print("DONE — all outputs saved to", OUTPUT_DIR)
    print("="*80)

if __name__ == '__main__':
    main()